In [1]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import re

In [2]:
df = pd.read_excel('/content/cases_train2 (15).xlsx')

In [3]:
texts = df['Решение кейса'].astype(str).str.replace(r'<br\s*/?>', ' ', regex=True)
tfidf_ref = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf_ref.fit(texts)
joblib.dump(tfidf_ref, 'tfidf_reference.joblib')

['tfidf_reference.joblib']

In [4]:
vec = joblib.load('tfidf_reference.joblib')

In [5]:
REFERENCES = {
    "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ МАЛОГО И СРЕДНЕГО БИЗНЕСА": (
        "Выбрал HoReCa: 72% безналичных платежей (ЦБ РФ, 2024), рост "
        "рынка 14% (RestCon 2024), 68% владельцев называют кассовые "
        "разрывы главной проблемой (собственный опрос). ЦА: сегмент 1 — "
        "микробизнес (1–2 точки, 1–5 чел., оборот 0,5–2 млн ₽/мес): "
        "сезонные разрывы −30–40%, потери от порчи 15%, комиссия "
        "эквайринга 2,2–2,5%; решают за 1–3 дня. Сегмент 2 — малый "
        "бизнес (1–3 точки, 10–50 чел., 2–10 млн ₽/мес): несколько "
        "потоков выручки, сложные расчёты с персоналом, дебиторка 30–45 "
        "дней; решают 1–2 недели. Сегмент 3 — сети (5+ точек, 50–200 "
        "чел., 10–50 млн ₽/мес): ликвидность между юрлицами, "
        "ERP-интеграция, факторинг; цикл 1–3 мес. CJM: рекомендация → "
        "демо → внедрение за 1–2 недели → веб-кабинет. Решение "
        "«Альфа.HoReCa»: овердрафт под эквайринг (до 5 млн ₽, 12–18%, "
        "одобрение за 2 ч); кредит на открытие (до 3 млн ₽, отсрочка 6 "
        "мес., 10%); факторинг (до 90%, 8–12%); сезонная линия (до 15 "
        "млн ₽). Нефинансовые: автозакупки с iiko/R-Keeper (потери "
        "−10–15%), безналичные чаевые (чек +9%, чаевые +23% в пилоте), "
        "авторасчёт зарплат (4 ч → 15 мин), юрбухсервисы, панель "
        "аналитики. Партнёры: iiko, R-Keeper, Loymax, «Лента Бизнес», "
        "«Яндекс.Еда», «АльфаСтрахование». Отличие: у Сбера сервисы "
        "разрознены, у Тинькофф нет автозакупок, у ВТБ нет нишевого "
        "решения. Финансы: CAPEX 50 млн ₽. Базовый сценарий: CAC 6,2 "
        "тыс. ₽, LTV 112 тыс. ₽ (LTV/CAC 18,1), 1 800→5 800 клиентов, "
        "выручка 67→216 млн ₽, NPV 156 млн ₽ (15%), IRR 34%, "
        "окупаемость 22 мес. Пессимистичный: окупаемость 32 мес., "
        "прибыль −24/−8/+18 млн ₽. Доходы: эквайринг, проценты, "
        "подписка (40%), партнёрские отчисления. Риски: тарифы ЦБ (40%) "
        "— диверсификация в подписку; низкая цифровая грамотность (60%) "
        "— обучение, CAC +15%; конкуренция (35%) — эксклюзив с iiko на "
        "2 года; сбои (15%) — подушка 4 млн ₽; спрос (30%) — фокус на "
        "доставку. Все риски учтены в сценариях. Доказательства: "
        "RestCon, ЦБ РФ, SmartRanking, Frank RG, 2ГИС, NielsenIQ; 15 "
        "интервью; 2 пилота (чаевые: чек +9%, закупки: время −40%, "
        "порча −12%); сравнение конкурентов по 8 параметрам."
    ),
    (
        "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ ОРГАНИЗАЦИЙ "
        "КУЛЬТУРНО-ДОСУГОВОГО НАЗНАЧЕНИЯ"
    ): (
        "Для проекта выбран ДК «Современник» (4500 м²) в спальном "
        "районе. Проведено исследование: 100 опросов, 25 интервью, "
        "данные Росстата и Минкульта. Построена сегментация по CJM: "
        "1. Семьи с детьми (боль — дорого в ТЦ, триггер — пакет «все "
        "включено» за 1500 руб. с кешбэком банка); 2. Подростки (боль — "
        "платные студии, триггер — бесплатные мастер-классы); 3. "
        "Взрослые 25-55 лет (боль — ДК закрыты рано, триггер — вечерние "
        "сеансы); 4. Арендаторы (нужна прозрачная аренда, триггер — "
        "личный кабинет). Решение — экосистема «Альфа-Культура 360»: "
        "1. Финансы (кредит 65 млн под 7,5%, РКО, эквайринг, факторинг); "
        "2. Цифра (приложение с ИИ, кешбэк, интеграция с Госуслугами); "
        "3. Нефинансовые сервисы (грантовый консалтинг, юридическая "
        "поддержка, маркетинг). Партнеры: Яндекс.Афиша, СберОбразование, "
        "сеть кофеен, IT-компания, администрация, 4 студии. "
        "Уникальность: объединение банковских продуктов, цифровой "
        "платформы и грантовой поддержки (у Сбера и Тинькофф этого нет). "
        "Финансы: Capex 100 млн (65 кредит + 25 грант + 10 инвестор). "
        "Выручка Y1: 30,9 млн, Y2: 49,3 млн, Y3: 68,4 млн. Чистая "
        "прибыль: Y1 –7,6 млн, Y2 6,8 млн, Y3 21,8 млн. DPP = 5,6 лет, "
        "NPV = 34,2 млн, IRR = 26,8%. CAC = 410 руб., LTV = 8200 руб. "
        "Три сценария (пессимистичный — окупаемость 7,1 лет). Риски: 8 "
        "рисков (рост цен на 55%, отказ в гранте 25%, низкая "
        "посещаемость 60% и др.) с вероятностями, решениями и учетом в "
        "финмодели. Подушка безопасности — 10 млн (10% от Capex). "
        "Доказательства: собственный пилот (pop-up кинотеатр — 680 "
        "зрителей, 84% готовы платить 400-600 руб); 25 интервью; опрос "
        "320 жителей (76% поддержали); 4 зарубежных кейса; анализ "
        "конкурентов; ссылки на Росстат и Минкульт; письма от партнеров."
    ),
    (
        "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ РАЗВИТИЯ МАССОВОГО СПОРТА "
        "И ФИЗИЧЕСКОЙ КУЛЬТУРЫ"
    ): (
        "ЦА: Трудовые мигранты (20–40 лет) и их семьи, студенты-"
        "иностранцы. По опросу (N=150, 3 района), 72% готовы "
        "заниматься при языковой поддержке (CJM: барьер языка/доверия → "
        "адаптация через куратора → покупка). Проблемы: недоверие к "
        "банкам, боязнь цен, языковой барьер. Решение «Альфа-Интеграция» "
        "(экосистема): 1. Мультиязычный хаб (4 языка) с гайдом и "
        "ИИ-переводчиком. 2. Тариф «Стартовый» (скидка 50% на 1-й месяц "
        "по патенту) + Семейный пакет (-30% от 3 человек). 3. "
        "Спорт-куратор (волонтёры диаспор) с бонусами за приведённых "
        "друзей (снимает риск недоверия). 4. Микрозаймы до 10 тыс. руб. "
        "на экипировку (льготный период 3 мес.). Партнёры: Агентство по "
        "делам национальностей (верификация), диаспоры (трафик), "
        "эконом-клубы, МФЦ. Уникальность: Банк — социальный "
        "интегратор. В отличие от Сбера (нет мультиязычности) и Т-Банка "
        "(нет офлайн-кураторов), мы даём комплексную адаптацию. Финансы "
        "(3 сценария): Бюджет: 150 млн (IT — 40, маркетинг — 25, фонд "
        "займов — 60, контент — 15, резерв — 10). Прогноз: 3к → 9к → "
        "20к клиентов. ARPU — 3 500 ₽. Выручка: 10,5/31,5/70 млн ₽. "
        "CAC — 2 800 ₽ (через диаспоры). LTV — 14 000 ₽ (LTV/CAC = 5). "
        "NPV (12%) = +18 млн ₽. Окупаемость — 2,9 года. Базовый "
        "сценарий — 60%, пессимистичный (отток 30%) — 25%, "
        "оптимистичный — 15%. Риски (с митигацией и учетом в модели): "
        "1. Недоверие (60%) — отзывы лидеров диаспор (учтено в "
        "маркетинге). 2. Язык (50%) — билингвы в колл-центре (+3 млн ₽ "
        "в модели). 3. Юридические (20%) — аудит миграционного права "
        "(резерв 5 млн ₽). 4. Отток (30%) — геймификация и кураторы "
        "(подушка 10 млн ₽). Доказательства: Собственный пилот в 2 "
        "клубах (N=100, конверсия 68%, NPS 8.5), опрос 150 мигрантов, "
        "данные УВМ МВД, A/B-тест SMS на 3 языках (+20% к "
        "кликабельности). Бенчмарк: кейс МТС-Банка (без спорт-вертикали)."
    ),
    (
        "КОМПЛЕКСНОЕ ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ СИСТЕМЫ "
        "ОБРАЗОВАНИЯ НА БАЗЕ ИСКУССТВЕННОГО ИНТЕЛЛЕКТА"
    ): (
        "Я выбрал обучение взрослых (финансовая грамотность для "
        "пенсионеров). ЦА: пенсионеры, их дети, социальные работники. "
        "Пенсионеры боятся мошенников и не понимают цифровые "
        "финансовые продукты, дети переживают за их сбережения, "
        "соцработники не знают, как доступно объяснять. Я предлагаю "
        "ИИ-ассистента с голосовым интерфейсом «Финансовый помощник»: "
        "он объясняет простыми словами, как работают вклады, кредиты, "
        "как не попасться на уловки мошенников. ИИ адаптирует скорость "
        "речи и сложность в зависимости от возраста пользователя. "
        "Уникальность — помощник работает через звонок (не через "
        "приложение, чтобы было удобно пожилым). Партнёры — Пенсионный "
        "фонд, центры социального обслуживания. Разработка — 15 млн, "
        "голосовой движок — 3 млн. Прогноз: 10 000 подписок по 1000 "
        "руб./год = 10 млн (не окупается). B2G: субсидии от регионов — "
        "30 млн. Итого 40 млн, окупаемость — 10 месяцев. Риски: "
        "пожилые не доверяют роботам, могут не понять. Решения: "
        "использовать голос известных дикторов (например, Левитан), "
        "проводить демо в центрах. Доказательства — я протестировал "
        "прототип на 30 пенсионерах, 25 сказали, что им понятнее, чем "
        "консультация в банке."
    ),
    "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ НЕФТЕГАЗОВОГО СЕКТОРА": (
        "Выбрал обучение взрослых (предпенсионеры 50-55 лет). ЦА — "
        "люди, которых сокращают. Боль — потеря работы, нужна новая "
        "профессия, но страшно начинать. Решение — ИИ-карьерный "
        "консультант, который анализирует прошлый опыт (диплом, стаж) и "
        "предлагает 3 профессии с планом переобучения (мини-курсы, "
        "стажировки). Уникальность — эмоциональная поддержка + "
        "геймификация (маленькие шаги). Партнёры — Центр занятости, "
        "hh.ru, Skillbox. Бюджет — 75 млн руб. (анализ резюме, "
        "ИИ-психолог). Планируем 50 000 пользователей (бесплатно через "
        "биржу труда), платные консультации — 500 руб. Доход — 15 млн "
        "руб. (конверсия 30%). Окупаемость — 5 лет (слишком долго). "
        "Риски: люди не доверяют ИИ в выборе карьеры. Доказательства: "
        "интервью с 30 предпенсионерами, статистика ЦЗН по сокращениям."
    ),
    (
        "ФИДЖИТАЛ-ОФИС 2.0 — НОВОЕ ПОКОЛЕНИЕ БАНКОВСКИХ ОТДЕЛЕНИЙ ДЛЯ "
        "ГОРОДОВ С НАСЕЛЕНИЕМ ОТ 30 ТЫСЯЧ ЧЕЛОВЕК"
    ): (
        "Тема: Фиджитал-офис-центр психологической поддержки в г. "
        "Архангельск (~350 тыс. чел.). ЦА: 4 сегмента: люди в "
        "кризисной ситуации (потеря работы, развод), предприниматели с "
        "выгоранием, пенсионеры с тревогой, студенты с паникой перед "
        "сессией. Проблемы: негде получить бесплатную консультацию, "
        "сложно найти деньги на лечение, ипотека давит. Решение: Офис с "
        "комнатами для разговора, биометрией для анонимного входа, "
        "зоной медитации и видеоконсультациями с психологами. Клиент "
        "может получить финансовую помощь (кредит/реструктуризацию) и "
        "психологическую поддержку в одном месте. Партнёры – Минздрав "
        "(психологи), кризисный центр, центр «Моя карьера». Финансы: "
        "CAPEX 20 млн руб. Доход от кредитов, страховок и партнёрских "
        "отчислений – 15 млн/год, окупаемость 2,2 года. Есть CAC, LTV, "
        "NPS (целевой 75), три сценария. Риски: Стигма – сделать "
        "анонимный вход и неонлайновую запись через МП. Отказ "
        "психологов – платить им выше рынка. Доказательства: Данные ВОЗ "
        "о тревожности в регионах (рост 20% в 2025), интервью с 50 "
        "жителями, кейс «Почта Банка» по социальным офисам, собственный "
        "A/B-тест анкеты."
    ),
    (
        "ЦЕНТР ПОДГОТОВКИ ПРОФЕССИОНАЛОВ БУДУЩЕГО — "
        "ОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ 2050"
    ): (
        "К 2050 г. беспилотный транспорт станет нормой, но дефицит "
        "инженеров ITS тормозит внедрение. По данным Минтранса, к 2035 "
        "г. нужно заменить 30% грузовиков, а вузы учат устаревшим "
        "системам. ЦА (CJM: поиск → обучение → внедрение): 1. Инженеры "
        "(25–40 лет): боятся устареть. Хотят перейти в ITS. 2. "
        "Студенты МАДИ (18–22): нет практики с реальными шасси. 3. "
        "Чиновники (30–50): не знают, как внедрять ИИ-решения в "
        "города. Решение (Экосистема): \"Альфа-Транспорт\" объединяет 3 "
        "модуля: 1. Магистратура + курсы (переподготовка). 2. Полигон с "
        "шасси КамАЗ. 3. ИИ-лаборатория (симуляция трафика, прогноз "
        "аварийности). Уникальность: не просто вуз, а закрытый цикл от "
        "обучения до внедрения в регионе. Партнёры: МАДИ (наука), КамАЗ "
        "(техника), Яндекс (софт), Минтранс (нормативы). Финансы (3 "
        "сценария): CAPEX: полигон (100 млн), ИИ (60 млн), шасси (30 "
        "млн). OPEX: 50 млн/год. 200 студентов (чек: магистры 600к, "
        "инженеры 400к, чиновники 200к). Выручка: 110 млн/год (базовый "
        "сценарий). Метрики: CAC — 15 тыс. руб., LTV — 1,2 млн руб., "
        "NPV за 5 лет — 45 млн руб., ROI — 40%. Окупаемость — 2,5 года "
        "(при базовом). Подушка — 15 млн руб. Риски: 1. Законы "
        "(высокий): Минтранс легализует — параллельные НИР. 2. "
        "Технологии (средний): сбои ИИ — дублирование систем. 3. "
        "Финансы (низкий): гранты Альфа-Банка. Доказательства: Провели "
        "пилот в Москве (ИИ-симуляция сократила пробки на 20%). "
        "Интервью с главой Яндекс.Беспилотников подтвердило спрос."
    ),
    (
        "ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ РАЗВИТИЯ "
        "ПРОФЕССИОНАЛЬНОГО СПОРТА В РЕГИОНЕ"
    ): (
        "Выбираю теннис в Краснодарском крае: спрос растёт, но высокая "
        "стоимость экипировки и кортов отсекает таланты. По данным "
        "Федерации тенниса края (2 500 юниоров), лишь 2% имеют "
        "спонсоров. Провёл опрос 100 родителей и 15 тренеров. Целевая "
        "аудитория: 4 сегмента с CJM: 1. Дети 7–10 лет (обеспеченные): "
        "Платят 50 тыс. руб./мес. без контроля эффективности. 2. Дети "
        "11–14 лет (средний класс): CJM: ребёнок выигрывает → нужны "
        "турниры (до 200 тыс. руб./мес.) → кредит в Сбере под 20% → "
        "разочарование → отсев. 3. Дети 15–17 лет: Нужны спонсоры. 4. "
        "Родители (ЛПР): Боль — непрозрачность трат и страх "
        "неокупаемости. Решение: Экосистема «Альфа-Теннис»: 1. Кредит "
        "на экипировку/поездки под 4% с льготным периодом 6 мес. "
        "(партнёры «Head», «Wilson»). 2. Гранты 30 юниорам по 150 тыс. "
        "руб./год. 3. AI-платформа отбора по видео (IT-партнёр). 4. "
        "Кешбэк 10% на корты/инвентарь. 5. Турнир с фондом 3 млн руб. и "
        "ТВ-трансляцией. 6. Приложение с учётом трат и прогнозом "
        "карьеры. Уникальность: создаём финансово-карьерный трек, в "
        "отличие от простого спонсорства конкурентов. Финансовая "
        "модель: Расходы: 24,5 млн руб. (кредитный риск 4, гранты 4,5, "
        "AI 6, турнир 3, приложение 3, маркетинг 4). Доходы: 10,7 млн "
        "руб./год (проценты 1,2, РКО 6, партнёры 2, спонсоры 1,5). "
        "Окупаемость: 3 года. CAC 25 000 руб., LTV 85 000 руб., ROI "
        "85%. Риски: Дефолт (средняя) — скоринг и страховка. Низкий "
        "охват AI (средняя) — обучение тренеров. Сбои (низкая) — "
        "резерв. Низкий интерес (низкая) — звёзды ATP. Подушка 15% в "
        "финмодели. Доказательства: Опрос 100 родителей и 15 тренеров. "
        "Кейс «Альфа-Банк × ЦСКА» (только спонсорство, без экосистемы). "
        "A/B-тест на 50 родителях: 70% готовы взять кредит под 4%."
    ),
    "ЭЛЕКТРОМОБИЛЬНЫЙ КЛАСТЕР ДЛЯ РЕГИОНА": (
        "Проект: Завод по выпуску электромобилей B-класса для такси и "
        "каршеринга. Анализ ЦА: Глубокая сегментация: Таксопарки "
        "(важна стоимость км, гарантия 300 тыс. км), Каршеринг "
        "(быстрая уборка, удаленный доступ), Частные покупатели "
        "(дизайн, подписка). Поведенческий анализ, тренды и CJM от "
        "тест-драйва до утилизации АКБ. Проработка решения: "
        "Электромобиль B-класса с ударопрочным кузовом и быстрой "
        "зарядкой (80% за 20 мин). Партнеры: Сети таксопарков и "
        "энергосбытовая компания. Финансовая модель: Capex — 7.2 млрд "
        "руб. Opex — 400 млн руб./год. Цена авто — 2.1 млн руб. CAC = "
        "42 тыс. руб., LTV = 310 тыс. руб. Потоки на 10 лет: NPV = "
        "1.85 млрд руб., IRR = 23.1%, WACC = 13%, DPBP = 4.9 года (3 "
        "детальных сценария). Анализ рисков: 1) Подорожание "
        "компонентов (вер. 35%) — долгосрочные контракты; 2) Низкий "
        "спрос зимой (вер. 20%) — внедрение термостатирования АКБ. В "
        "финмодель заложен резервный фонд рисков — 550 млн руб. (7.6% "
        "Capex). Доказательства: Ссылки на собственное глубинное "
        "интервью с 10 владельцами таксопарков и результаты пилотного "
        "опроса 600 водителей."
    ),
    "АНТИКРИЗИСНАЯ СТРАТЕГИЯ ДЛЯ СЕТИ РЕСТОРАНОВ": (
        "Полностью проработанная антикризисная стратегия. Анализ ЦА и "
        "рынка: проведен комплексный анализ динамики рынка HoReCa. "
        "Аудитория сегментирована с детальной проработкой CJM и "
        "поведенческих факторов: деловой сегмент (40%) — боли: срыв "
        "сроков подачи, падение качества ланчей; семейный премиум "
        "(35%) — боли: отсутствие детской анимации, изменение "
        "привычного вкуса блюд; персонал — выгорание, падение доходов "
        "из-за снижения чека и чаевых. Проработка решения: предложен "
        "комплексный экосистемный подход — интеграция офлайн и онлайн: "
        "Ресторан + приложение с персональным шеф-меню, предзаказом и "
        "подпиской; Академия сервиса — возвращение учебного центра с "
        "грейдовой системой оплаты труда; сеть партнеров — локальные "
        "эко-фермы, премиальные сервис-консьержи и винодельни. "
        "Финансовая модель и метрики: инвестиции 7,8 млн руб. (IT, "
        "маркетинг, ФОТ, обновление ТТК блюд). Метрики: NPV = 15,4 млн "
        "руб., IRR = 31%, CAC = 1 750 руб., LTV = 58 000 руб. Срок "
        "окупаемости — 9 месяцев. Просчитаны пессимистичный, базовый и "
        "оптимистичный сценарии развития. Анализ рисков: проведена "
        "оценка вероятности возникновения рисков. В финансовую модель "
        "официально включена подушка безопасности (10% от ежемесячных "
        "операционных расходов) для покрытия риска инфляции сырья и "
        "кадровых издержек. Доказательная база: решение опирается на "
        "собственное исследование (CustDev-интервью с 40 бывшими "
        "клиентами сети) и результаты A/B тестирования концепции "
        "обновления сервиса."
    ),
}

In [6]:
text_reference = (
    "Выбрал HoReCa: 72% безналичных платежей (ЦБ РФ, 2024), рост "
    "рынка 14% (RestCon 2024), 68% владельцев называют кассовые "
    "разрывы главной проблемой (собственный опрос). ЦА: сегмент 1 — "
    "микробизнес (1–2 точки, 1–5 чел., оборот 0,5–2 млн ₽/мес): "
    "сезонные разрывы −30–40%, потери от порчи 15%, комиссия "
    "эквайринга 2,2–2,5%; решают за 1–3 дня. Сегмент 2 — малый "
    "бизнес (1–3 точки, 10–50 чел., 2–10 млн ₽/мес): несколько "
    "потоков выручки, сложные расчёты с персоналом, дебиторка 30–45 "
    "дней; решают 1–2 недели. Сегмент 3 — сети (5+ точек, 50–200 "
    "чел., 10–50 млн ₽/мес): ликвидность между юрлицами, "
    "ERP-интеграция, факторинг; цикл 1–3 мес. CJM: рекомендация → "
    "демо → внедрение за 1–2 недели → веб-кабинет. Решение "
    "«Альфа.HoReCa»: овердрафт под эквайринг (до 5 млн ₽, 12–18%, "
    "одобрение за 2 ч); кредит на открытие (до 3 млн ₽, отсрочка 6 "
    "мес., 10%); факторинг (до 90%, 8–12%); сезонная линия (до 15 "
    "млн ₽). Нефинансовые: автозакупки с iiko/R-Keeper (потери "
    "−10–15%), безналичные чаевые (чек +9%, чаевые +23% в пилоте), "
    "авторасчёт зарплат (4 ч → 15 мин), юрбухсервисы, панель "
    "аналитики. Партнёры: iiko, R-Keeper, Loymax, «Лента Бизнес», "
    "«Яндекс.Еда», «АльфаСтрахование». Отличие: у Сбера сервисы "
    "разрознены, у Тинькофф нет автозакупок, у ВТБ нет нишевого "
    "решения. Финансы: CAPEX 50 млн ₽. Базовый сценарий: CAC 6,2 "
    "тыс. ₽, LTV 112 тыс. ₽ (LTV/CAC 18,1), 1 800→5 800 клиентов, "
    "выручка 67→216 млн ₽, NPV 156 млн ₽ (15%), IRR 34%, "
    "окупаемость 22 мес. Пессимистичный: окупаемость 32 мес., "
    "прибыль −24/−8/+18 млн ₽. Доходы: эквайринг, проценты, "
    "подписка (40%), партнёрские отчисления. Риски: тарифы ЦБ (40%) "
    "— диверсификация в подписку; низкая цифровая грамотность (60%) "
    "— обучение, CAC +15%; конкуренция (35%) — эксклюзив с iiko на "
    "2 года; сбои (15%) — подушка 4 млн ₽; спрос (30%) — фокус на "
    "доставку. Все риски учтены в сценариях. Доказательства: "
    "RestCon, ЦБ РФ, SmartRanking, Frank RG, 2ГИС, NielsenIQ; 15 "
    "интервью; 2 пилота (чаевые: чек +9%, закупки: время −40%, "
    "порча −12%); сравнение конкурентов по 8 параметрам."
)
text_not_renerence = (
    "Наша инновационная экосистема гарантированно обеспечит рост "
    "выручки сети на 45% в первый год за счёт внедрения омниканальной "
    "модели, персонализированного клиентского опыта и предиктивной "
    "аналитики. Данное решение является прорывным и не имеет аналогов "
    "на рынке. Эффективность очевидна, поскольку основана на лучших "
    "мировых практиках. Целевая аудитория — все гости ресторанов. Мы "
    "не делим их на группы, поскольку наша экосистема универсальна и "
    "подходит каждому. Гости хотят вкусно есть и получать "
    "удовольствие — это их главная потребность. Внедряем CRM, "
    "приложение, программу лояльности, AI-рекомендации. Это "
    "уникально, потому что никто в нашем городе такого не делал. "
    "Партнёры — крупные IT-компании. Инвестиции составят 10 млн "
    "рублей. Рост выручки на 45% очевиден благодаря синергии каналов. "
    "Окупаемость — 6 месяцев. Расчёты не привожу, так как они "
    "основаны на общеизвестных рыночных данных. Рисков нет, потому "
    "что решение проверено мировыми практиками. Мы уверены в успехе. "
    "Мы изучили опыт Starbucks и McDonald's. Их успех доказывает, что "
    "наша модель сработает. Ссылок нет."
)

In [7]:
# Функция проверки является ли решение эталонным
ref_vectors = {name: vec.transform([ref]) for name, ref in REFERENCES.items()}
REFERENCE_THRESHOLD = 0.9
def check_not_reference(text, case_name):
  # возращает 0 если решение является эталонным и его нельзя доаускать, иначе возращает 1
    if case_name not in ref_vectors:
        return 1
    text = re.sub(r"<br\s*/?>", " ", str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    similarity = (vec.transform([text]) @ ref_vectors[case_name].T).toarray()[0, 0]
    return int(similarity < REFERENCE_THRESHOLD)

In [9]:
check_not_reference(text_reference,"ОТРАСЛЕВОЕ РЕШЕНИЕ АЛЬФА-БАНКА ДЛЯ МАЛОГО И СРЕДНЕГО БИЗНЕСА")

0